# Readme

what this section is about.


# Imports and Load Data

In [59]:
## load packages 
import jieba
import jieba.posseg as pseg
import jieba.analyse
import pandas as pd
import re
import numpy as np
import os
import ast
import pprint


## sklearn imports
from sklearn.feature_extraction.text import CountVectorizer

## lda
from gensim import corpora
import gensim

## visualizing LDA
import pyLDAvis.gensim_models as gensimvis
import pyLDAvis
pyLDAvis.enable_notebook()

## plotting
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.font_manager as fm

_cjk = ['PingFang SC', 'Heiti SC', 'STHeiti', 'SimHei', 'Noto Sans CJK SC']
_avail = {f.name for f in fm.fontManager.ttflist}
_font = next((f for f in _cjk if f in _avail), None)
if _font:
    matplotlib.rcParams['font.family'] = _font
matplotlib.rcParams['axes.unicode_minus'] = False

## print mult things
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [60]:
df = pd.read_csv("../data/01_filtered_cases.csv")
df.head()
df.shape

,case_name,case_type,court,date,case_href,full_text,litigant_type
0,刘慷、吉林省射击射箭运动管理中心人事争议民事再审民事裁定书,民事再审,吉林省高级人民法院,2021-11-05,https://wenshu.court.gov.cn/website/wenshu/181...,吉林省高级人民法院\n民 事 裁 定 书\n（2021）吉民再270号\n再审申请人（一审原...,individual
1,刘小宝、赵效男等买卖合同纠纷民事申请再审审查民事裁定书,民事再审,山东省高级人民法院,2021-07-05,https://wenshu.court.gov.cn/website/wenshu/181...,山东省高级人民法院\n民 事 裁 定 书\n（2021）鲁民申3670号\n再审申请人（一审...,individual
2,马晓三、云南省人民检察院劳动争议再审民事判决书,民事再审,云南省高级人民法院,2020-06-09,https://wenshu.court.gov.cn/website/wenshu/181...,云南省高级人民法院\n民 事 判 决 书\n（2019）云民再30号\n抗诉机关：云南省人民...,individual
3,李素萍与北京市木樨园体育运动技术学校劳动争议再审审查与审判监督民事裁定书,民事审判监督,北京市高级人民法院,2020-06-05,https://wenshu.court.gov.cn/website/wenshu/181...,北京市高级人民法院\n民 事 裁 定 书\n（2020）京民申2236号\n再审申请人（一审...,individual
4,马俊岭等十三人故意伤害二审刑事附带民事判决书,刑事二审,陕西省高级人民法院,2018-07-30,https://wenshu.court.gov.cn/website/wenshu/181...,陕西省高级人民法院\n刑 事 附 带 民 事 判 决 书\n（2018）陕刑终147号\n原...,individual


(79, 7)

# preprocess text


## 1. define stopwords for text cleaning

In [61]:
# Strip out : general Chinese stopwords + 
# legal boilerplate and procedural terms to strip before topic modeling
# Modified: 11 boilerplate terms added after LDA output review (民事判决, 民事裁定,
#   民事, 出生, 汉族, 送达, 公告送达, 案件, 申请, 规定, 京民)
STOPWORDS = set("""
一审 二审 再审 原告 被告 申请人 被申请人 上诉人 被上诉人 驳回上诉 诉至 抗诉 审判长 审判员
书记员 院长 副院长 庭长 法官 助理 合议庭 人民法院 高级人民法院 中级人民法院 基层人民法院
判决 裁定 裁决 判决书 裁定书 法律 如下 认为 依照 根据 查明 本院 经审查 经审理 经查 裁决书 检察院 检察员
中华人民共和国 人民币 甲方 乙方 丙方 甲乙双方 本院认为 无证据 诉讼法 律师费 民法典 公告费 不服 传唤 案外人 给付 
举办方 本院认为 原审 贵院 实缴 该证据 受理费 住所地 辩称 上诉状 判令 载明 诉讼请求 简易程序 采信 起诉 公证费
审批表 诉称 综上 副本 举证 自认 诉请 人民陪审员 开庭 受案 上述事实 本合同 应予 处理结果 不予 提出 证据
答辩 维持原判 驳回 纠纷 驳回 委托 审查 被执行人 撤诉 答辩 终号 因案 维持原判 发回重审 终审 案号 诉讼费用 在案
委托 诉讼 代理人 律师 事务所 法定代表 事务所律师 上诉 立案 依法 组成 进行 审理 本案 现已 终结 请求 依法 一审判决 
第一 判项 改判 诉讼费 事实 理由 承担 
理由 法院 最高人民法院 关于 若干 问题 指导 意见 第三条 第二项 原名 称为 不可 不遗余力 后于
出生 送达 公告送达 案件 申请 规定 京民
""".split())

print(f"Stopword list size: {len(STOPWORDS)}")

Stopword list size: 152


## 2. tokenize

In [62]:
def clean_text(text):
    # keep only chinese characters
    text = re.sub(r'[^一-鿿]', '', text)
    lwords = jieba.lcut(text)
    lwords = [w for w in lwords if w not in STOPWORDS and len(w) > 1]
    # de-dup
    lwords_uniq = list(set(lwords))
    
    return "".join(lwords_uniq)

In [63]:
# # pos checker
# words = pseg.cut("综上" ) #paddle模式
# for word, flag in words:
#     print('%s %s' % (word, flag))

In [64]:
df["keywords"] = df["full_text"].apply(clean_text).apply(
    lambda sentence: jieba.analyse.extract_tags(
        sentence, 
        topK=50,
        withWeight=False, 
        allowPOS=('n', 'nt', 'nz', 'v', 'vn', 'a')
    )
)

df.head()

,case_name,case_type,court,date,case_href,full_text,litigant_type,keywords
0,刘慷、吉林省射击射箭运动管理中心人事争议民事再审民事裁定书,民事再审,吉林省高级人民法院,2021-11-05,https://wenshu.court.gov.cn/website/wenshu/181...,吉林省高级人民法院\n民 事 裁 定 书\n（2021）吉民再270号\n再审申请人（一审原...,individual,"[吉体, 劳动厅, 补上去, 航模, 人事厅, 劳动局, 原省, 吉民, 绿园区, 销应, ..."
1,刘小宝、赵效男等买卖合同纠纷民事申请再审审查民事裁定书,民事再审,山东省高级人民法院,2021-07-05,https://wenshu.court.gov.cn/website/wenshu/181...,山东省高级人民法院\n民 事 裁 定 书\n（2021）鲁民申3670号\n再审申请人（一审...,individual,"[历下区, 万床, 男因, 海蝶, 柴家, 书鲁民, 称有, 申号, 工商登记, 市中区, ..."
2,马晓三、云南省人民检察院劳动争议再审民事判决书,民事再审,云南省高级人民法院,2020-06-09,https://wenshu.court.gov.cn/website/wenshu/181...,云南省高级人民法院\n民 事 判 决 书\n（2019）云民再30号\n抗诉机关：云南省人民...,individual,"[省体, 法制报, 云南省人民政府, 权利, 奖惩条例, 国家经贸委, 申诉人, 待岗, 政..."
3,李素萍与北京市木樨园体育运动技术学校劳动争议再审审查与审判监督民事裁定书,民事审判监督,北京市高级人民法院,2020-06-05,https://wenshu.court.gov.cn/website/wenshu/181...,北京市高级人民法院\n民 事 裁 定 书\n（2020）京民申2236号\n再审申请人（一审...,individual,"[人事科, 周润, 体育训练, 意见栏, 木樨园, 京民, 原系, 套改, 表中, 书京民,..."
4,马俊岭等十三人故意伤害二审刑事附带民事判决书,刑事二审,陕西省高级人民法院,2018-07-30,https://wenshu.court.gov.cn/website/wenshu/181...,陕西省高级人民法院\n刑 事 附 带 民 事 判 决 书\n（2018）陕刑终147号\n原...,individual,"[杯间, 枕部, 监控室, 挫裂伤, 碑林区, 侧脑室, 酒吧女, 塑料箱, 鉴定费, 库门..."


In [65]:
# export df with tokenized keywords for later analysis

df.to_csv("../data/02_tokenized.csv", index=False)
print(f"Saved to {os.path.abspath('../data/')}")

Saved to /Users/wyx/Library/CloudStorage/OneDrive-DartmouthCollege/qss20_athlete_court_complaints/data


# Unsupervised topic modeling via Gensim

In [66]:
# keywords column already in lists format from cleaning step
df["keywordsl"] = df["keywords"]

In [67]:
## Step 1: re-tokenize and store in list
## here, i'm doing with the raw random sample of text
## in activity, you should do with the preprocessed texts
text_raw_tokens = [one_text for one_text in df.keywordsl]

## Step 2: use gensim create dictionary - gets all unique words across documents
text_raw_dict = corpora.Dictionary(text_raw_tokens)
raw_len = len(text_raw_dict) # get length for comparison below

### explore first few keys and values
### see that key is just an arbitrary counter; value is the word itself
{k: text_raw_dict[k] for k in list(text_raw_dict)[:5]}


## Step 3: filter out very rare and very common words
## here, i'm using the threshold that a word needs to appear in at least
## 5% of docs but not more than 95%
## this is an integer count of docs so i round
lower_bound = round(df.shape[0]*0.05)
upper_bound = round(df.shape[0]*0.95)

### apply filtering to dictionary
text_raw_dict.filter_extremes(no_below = lower_bound,
                             no_above = upper_bound)
print(f'Filtering out very rare and very common words reduced the \
length of dictionary from {str(raw_len)} to {str(len(text_raw_dict))}.')
{k: text_raw_dict[k] for k in list(text_raw_dict)[:5]} # show first five entries after filtering


## Step 4: apply dictionary to TOKENIZED texts
## this creates a mapping between each word 
## in a specific listing and the key in the dictionary.
## for words that remain in the filtered dictionary,
## output is a list where len(list) == n documents
## and each element in the list is a list of tuples
## containing the mappings
corpus_fromdict = [text_raw_dict.doc2bow(one_text) 
                   for one_text in text_raw_tokens]

### can apply doc2bow(one_text, return_missing = True) to print words
### eliminated from the listing bc they're not in filtered dictionary.
### but feeding that one with missing values to
### the lda function can cause errors
corpus_fromdict_showmiss = [text_raw_dict.doc2bow(one_text, return_missing = True)
                            for one_text in text_raw_tokens]
print('Sample of documents represented in dictionary format (with omitted words noted):')
corpus_fromdict_showmiss[:10]

{0: '人事厅', 1: '人事处', 2: '人员名单', 3: '人王', 4: '体委'}

Filtering out very rare and very common words reduced the length of dictionary from 1988 to 215.


{0: '人事厅', 1: '人事处', 2: '体委', 3: '体工', 4: '体工队'}

Sample of documents represented in dictionary format (with omitted words noted):


[([(0, 1),
   (1, 1),
   (2, 1),
   (3, 1),
   (4, 1),
   (5, 1),
   (6, 1),
   (7, 1),
   (8, 1),
   (9, 1),
   (10, 1),
   (11, 1),
   (12, 1),
   (13, 1),
   (14, 1),
   (15, 1),
   (16, 1),
   (17, 1)],
  {'人员名单': 1,
   '人王': 1,
   '体育事业': 1,
   '劳动厅': 1,
   '合法利益': 1,
   '吉体': 1,
   '吉民': 1,
   '名册': 1,
   '团体冠军': 1,
   '委以': 1,
   '工作失误': 1,
   '工资表': 1,
   '开庭审理': 1,
   '慷本': 1,
   '手写': 1,
   '扩大会': 1,
   '提审': 1,
   '文龙': 1,
   '机构编制': 1,
   '现住': 1,
   '离校': 1,
   '红头文件': 1,
   '绿园区': 1,
   '航模': 1,
   '补上去': 1,
   '表等': 1,
   '证据不足': 1,
   '调令': 1,
   '转业': 1,
   '转递': 1,
   '销应': 1,
   '龙运': 1}),
 ([(6, 1), (14, 1), (15, 1), (18, 1), (19, 1), (20, 1), (21, 1)],
  {'万床': 1,
   '书鲁民': 1,
   '供货': 1,
   '公司法': 1,
   '公示': 1,
   '出资': 1,
   '办事处': 1,
   '占用': 1,
   '历下区': 1,
   '受害者': 1,
   '可知': 1,
   '商贸': 1,
   '增资': 1,
   '实质性': 1,
   '工商登记': 1,
   '市中区': 1,
   '当事人': 1,
   '录音带': 1,
   '挫折': 1,
   '推翻': 1,
   '提交': 1,
   '文化课': 1,
   '无关': 1,
   '柴家': 1,
   '法定': 1,
   '注册

In [68]:
## Step 5: we're finally ready to estimate the model!
## full documentation here - https://radimrehurek.com/gensim/models/ldamodel.html
## here, we're feeding the lda function:
## (1) the corpus we created from the dictionary,
## (2) a parameter we decide on for the number of topics (k),
## (3) the dictionary itself,
## (4) parameter for number of passes through training data (more means slower), and
## (5) parameter that returns, for each word remaining in dict, the topic probabilities.
## see documentation for many other arguments you can vary
ldamod = gensim.models.ldamodel.LdaModel(corpus_fromdict,
                                         num_topics = 5, 
                                         id2word=text_raw_dict, 
                                         passes=15, 
                                         alpha ='auto',
                                         per_word_topics = True,
                                        random_state=42)

print(type(ldamod))

<class 'gensim.models.ldamodel.LdaModel'>


## Seeing what topics the estimated model discovers

In [69]:
## Post-model 1: explore corpus-wide summary of topics
### getting the topics and top words; can retrieve diff top words
topics = ldamod.print_topics(num_words = 10)
for topic in topics:
    print(topic)

(0, '0.036*"护理费" + 0.035*"住宿费" + 0.034*"营养费" + 0.029*"预交" + 0.028*"补偿金" + 0.024*"椎间盘" + 0.024*"补助费" + 0.024*"体工队" + 0.023*"工作队" + 0.023*"鉴定费"')
(1, '0.037*"体育局" + 0.031*"运动队" + 0.023*"工资待遇" + 0.023*"择业" + 0.022*"人事厅" + 0.021*"补偿金" + 0.020*"人事科" + 0.019*"人事处" + 0.018*"补发" + 0.017*"离队"')
(2, '0.029*"民事" + 0.029*"退役" + 0.027*"运动员" + 0.024*"民事裁定" + 0.022*"体育局" + 0.019*"微信" + 0.018*"补偿费" + 0.016*"截图" + 0.016*"传票" + 0.016*"抚慰金"')
(3, '0.023*"转款" + 0.022*"欠付" + 0.022*"出借" + 0.022*"履行合同" + 0.019*"公章" + 0.019*"补救措施" + 0.019*"公司财务" + 0.019*"借条" + 0.019*"专用章" + 0.019*"授权书"')
(4, '0.028*"民事判决" + 0.026*"体育局" + 0.025*"主体资格" + 0.025*"择业" + 0.023*"发号" + 0.021*"退役费" + 0.019*"人事部" + 0.019*"人事部门" + 0.018*"人事厅" + 0.017*"国家体育总局"')


### Visualization 

In [56]:
lda_display = gensimvis.prepare(ldamod, corpus_fromdict, text_raw_dict)
pyLDAvis.display(lda_display)

In [58]:
# as HTML file
pyLDAvis.save_html(lda_display, '../output/lda_vis_chn.html')